In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')

In [ ]:
matches = pd.read_csv("data/matches.csv")
deliveries = pd.read_csv("data/deliveries.csv")

print(matches.shape)
print(deliveries.shape)

matches.head()

In [ ]:
top_runs = (
    deliveries.groupby('batter')['batsman_runs']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(top_runs)

In [ ]:
plt.figure(figsize=(10,6))
top_runs.plot(kind='bar')

plt.title("Top 10 IPL Run Scorers")
plt.xlabel("Player")
plt.ylabel("Runs")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
player_stats = deliveries.groupby('batter').agg(
    Runs=('batsman_runs', 'sum'),
    Balls=('ball', 'count')
)

player_stats = player_stats[player_stats['Balls'] >= 500]

player_stats['Strike Rate'] = (
    player_stats['Runs'] /
    player_stats['Balls']
) * 100

top_sr = player_stats.sort_values(
    by='Strike Rate',
    ascending=False
).head(10)

print(top_sr)

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    x=top_sr.index,
    y=top_sr['Strike Rate']
)

plt.title("Top IPL Strike Rates")
plt.xticks(rotation=45)

plt.show()

In [ ]:
wins = matches['winner'].value_counts()

team_matches = pd.concat([
    matches['team1'],
    matches['team2']
]).value_counts()

team_stats = pd.DataFrame({
    'Matches Played': team_matches,
    'Wins': wins
}).fillna(0)

team_stats['Win %'] = (
    team_stats['Wins'] /
    team_stats['Matches Played']
) * 100

team_stats = team_stats.sort_values(
    by='Win %',
    ascending=False
)

print(team_stats)

In [ ]:
plt.figure(figsize=(12,6))

sns.barplot(
    x=team_stats.index,
    y=team_stats['Win %']
)

plt.title("IPL Team Win Percentage")
plt.xticks(rotation=75)

plt.show()

In [ ]:
match_season = matches[['id', 'season']]

merged = deliveries.merge(
    match_season,
    left_on='match_id',
    right_on='id'
)

merged.head()

In [ ]:
season_runs = (
    merged.groupby(['season', 'batter'])
    ['batsman_runs']
    .sum()
    .reset_index()
)

orange_caps = season_runs.loc[
    season_runs.groupby('season')
    ['batsman_runs']
    .idxmax()
]

orange_caps = orange_caps.sort_values('season')

print(orange_caps)

In [ ]:
plt.figure(figsize=(12,6))

sns.lineplot(
    data=orange_caps,
    x='season',
    y='batsman_runs',
    marker='o'
)

plt.title("Orange Cap Runs By Season")
plt.ylabel("Runs")

plt.show()

In [ ]:
virat = merged[
    merged['batter'] == 'V Kohli'
]

virat_season = (
    virat.groupby('season')
    ['batsman_runs']
    .sum()
    .reset_index()
)

print(virat_season)

In [ ]:
plt.figure(figsize=(10,5))

sns.lineplot(
    data=virat_season,
    x='season',
    y='batsman_runs',
    marker='o'
)

plt.title("Virat Kohli Runs Across IPL Seasons")
plt.ylabel("Runs")

plt.show()

In [ ]:
orange_caps.to_csv(
    "orange_cap_winners.csv",
    index=False
)

team_stats.to_csv(
    "team_win_rates.csv"
)

print("Files exported successfully.")

In [ ]:
players = [
    'V Kohli',
    'RG Sharma',
    'DA Warner',
    'MS Dhoni',
    'AB de Villiers'
]

player_comparison = merged[
    merged['batter'].isin(players)
]

season_runs = (
    player_comparison
    .groupby(['season', 'batter'])['batsman_runs']
    .sum()
    .reset_index()
)

In [ ]:
plt.figure(figsize=(12,6))

sns.lineplot(
    data=season_runs,
    x='season',
    y='batsman_runs',
    hue='batter',
    marker='o'
)

plt.title('Player Performance Comparison Across Seasons')
plt.show()